# Semantic Segmentation in Pontryagin Spaces — Colab Experiments

**Structure**
```
0. Setup & GPU check
1. Mount Google Drive
2. Clone repo & install package
3. Dataset
4. FCN backbone experiments
   4a. HPO  (optional, ~2-4 h per model)
   4b. Full training
   4c. Interpretability (SegScoreCAM + embedding energy)
   4d. Comparison
5. UNet backbone experiments
   5a. HPO  (optional)
   5b. Full training
   5c. Interpretability
   5d. Comparison
6. Download results
```

> **Before running:** set `REPO_URL`, `DATA_ZIP`, `DRIVE_DATA_ROOT`, and `DRIVE_RESULTS_ROOT` in cell **1b**.

## 0 · GPU check

In [1]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

CUDA available : True
GPU            : NVIDIA A100-SXM4-80GB
VRAM (GB)      : 85.1


## 1 · Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
REPO_URL         = 'https://github.com/craljimenez/rr-pontryagin-embedded.git'

# Path to UAV_segmantation.zip already uploaded to Drive
# (or leave empty to upload manually in cell 3)
DATA_ZIP         = '/content/drive/MyDrive/UAV_segmantation.zip'

# Where the unzipped dataset will live
DRIVE_DATA_ROOT  = '/content/drive/MyDrive/____UTP/Research/PHd/RFF_activation/UAV_segmantation'

# Where all experiment outputs will be saved (persists across sessions)
DRIVE_FCN_RESULTS  = '/content/drive/MyDrive/____UTP/Research/PHd/___RR_Pontryagin_embedded/results/fcn'
DRIVE_UNET_RESULTS = '/content/drive/MyDrive/____UTP/Research/PHd/___RR_Pontryagin_embedded/results/unet'
# ──────────────────────────────────────────────────────────────────────────────

import os
os.makedirs(DRIVE_FCN_RESULTS,  exist_ok=True)
os.makedirs(DRIVE_UNET_RESULTS, exist_ok=True)
print('Paths set.')

Paths set.


## 2 · Clone repo & install package

In [4]:
import os
REPO_DIR = '/content/pontryagin'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

Cloning into '/content/pontryagin'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 85 (delta 17), reused 74 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 2.38 MiB | 9.47 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/pontryagin


In [5]:
# Install the prfe package in editable mode + any missing deps
!pip install -e . -q
!pip install scikit-optimize -q
print('Package installed.')

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for prfe (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 10.8 MB/s eta 0:00:00
Package installed.


## 3 · Dataset

In [6]:
import os, zipfile

if not os.path.exists(DRIVE_DATA_ROOT):
    if os.path.exists(DATA_ZIP):
        print('Unzipping dataset from Drive …')
        with zipfile.ZipFile(DATA_ZIP, 'r') as z:
            z.extractall(os.path.dirname(DRIVE_DATA_ROOT))
        print('Done.')
    else:
        # Manual upload fallback
        from google.colab import files
        print('Upload UAV_segmantation.zip manually:')
        uploaded = files.upload()
        zname = list(uploaded.keys())[0]
        with zipfile.ZipFile(zname, 'r') as z:
            z.extractall(os.path.dirname(DRIVE_DATA_ROOT))
        print('Done.')
else:
    print('Dataset already exists:', DRIVE_DATA_ROOT)

# Verify splits
for split in ('train', 'valid', 'test'):
    n = len(list(__import__('pathlib').Path(DRIVE_DATA_ROOT, split, 'images').glob('*.jpg')))
    print(f'  {split:6s}: {n} images')

Dataset already exists: /content/drive/MyDrive/____UTP/Research/PHd/RFF_activation/UAV_segmantation
  train : 81 images
  valid : 30 images
  test  : 20 images


## 4 · FCN backbone experiments
Models: `euclidean` | `hyperbolic` | `pontryagin`

### 4a · HPO  *(optional — skips to 4b if best_params.json exists)*

In [7]:
# Run HPO for one model at a time.  ~2–4 h for 30 trials on a T4.
# FCN_MODEL = 'pontryagin'   # change to 'hyperbolic' or 'euclidean' as needed
for FCN_MODEL in ['euclidean', 'hyperbolic', 'pontryagin']:
    !python experiments/tune_seg_uav.py \
        --model        {FCN_MODEL} \
        --n-calls      30 \
        --trial-epochs 15 \
        --data-root    "{DRIVE_DATA_ROOT}" \
        --results-dir  "{DRIVE_FCN_RESULTS}"

Train: 49 | Valid: 23 | Test: 13

Search space for euclidean:
  lr               → Real(low=1e-05, high=0.01, prior='log-uniform', transform='identity')
  weight_decay     → Real(low=1e-06, high=0.001, prior='log-uniform', transform='identity')
  backbone_depth   → Integer(low=3, high=5, prior='uniform', transform='identity')

Starting forest_minimize (30 trials, 10 random) …
Iteration No: 1 started. Evaluating function at random point.

════════════════════════════════════════════════════════════
Trial   1 / 30  (euclidean)  →  trial_001
  lr = 0.0024526
  weight_decay = 3.5506e-06
  backbone_depth = 3
════════════════════════════════════════════════════════════
  → val macro mIoU = 0.7459  (85.9s)
Iteration No: 1 ended. Evaluation done at random point.
Time taken: 85.9564
Function value obtained: -0.7459
Current minimum: -0.7459
Iteration No: 2 started. Evaluating function at random point.

════════════════════════════════════════════════════════════
Trial   2 / 30  (euclidean)  →  t

### 4b · Full training  (60 epochs, uses best_params.json if present)

In [7]:
# Train all three FCN models sequentially.
# To train only one: change 'all' to 'euclidean', 'hyperbolic', or 'pontryagin'.
for model in ['euclidean', 'hyperbolic', 'pontryagin']:
    !python experiments/run_seg_uav.py \
        --model            {model} \
        --load-best-params \
        --data-root        "{DRIVE_DATA_ROOT}" \
        --results-dir      "{DRIVE_FCN_RESULTS}"

Loaded tuned params from /content/drive/MyDrive/____UTP/Research/PHd/___RR_Pontryagin_embedded/results/fcn/euclidean/hpo/best_params.json:
  lr = 0.0018260683738973073
  weight_decay = 0.0004124248727107401
  backbone_depth = 4
  _best_val_macro_iou = 0.8134175239143387
Model : euclidean
Device: cuda
Output: /content/drive/MyDrive/____UTP/Research/PHd/___RR_Pontryagin_embedded/results/fcn/euclidean
Train: 49 | Valid: 23 | Test: 13
Trainable parameters: 1,189,730  |  lr=1.83e-03  wd=4.12e-04

 Epoch      Loss    PixAcc      mIoU      Dice          LR
──────────────────────────────────────────────────────────
     1    0.7453    0.3964    0.1982    0.2839    1.82e-03  (29.4s)
  ↑ best model saved (macro mIoU = 0.1982)
     2    0.5815    0.6129    0.3174    0.4038    1.82e-03  (4.7s)
  ↑ best model saved (macro mIoU = 0.3174)
     3    0.4980    0.6230    0.3339    0.4310    1.81e-03  (4.7s)
  ↑ best model saved (macro mIoU = 0.3339)
     4    0.4584    0.6462    0.4014    0.5391    1.81

### 4c · Interpretability (SegScoreCAM + embedding energy + MC Dropout)

In [9]:
!python experiments/interpretability_analysis.py \
    --data-root   "{DRIVE_DATA_ROOT}" \
    --results-dir "{DRIVE_FCN_RESULTS}"

Using dataset root: /content/drive/MyDrive/____UTP/Research/PHd/RFF_activation/UAV_segmantation
Test split: 20 images
Skipping euclidean: Checkpoint not found: /content/drive/MyDrive/____UTP/Research/PHd/___RR_Pontryagin_embedded/results/fcn/euclidean/best_model.pth
Skipping hyperbolic: Checkpoint not found: /content/drive/MyDrive/____UTP/Research/PHd/___RR_Pontryagin_embedded/results/fcn/hyperbolic/best_model.pth
Skipping pontryagin: Checkpoint not found: /content/drive/MyDrive/____UTP/Research/PHd/___RR_Pontryagin_embedded/results/fcn/pontryagin/best_model.pth
Done.


In [10]:
# Embedding energy + MC Dropout (Pontryagin only)
!python experiments/embedding_energy.py \
    --data-root   "{DRIVE_DATA_ROOT}" \
    --results-dir "{DRIVE_FCN_RESULTS}"

Loading Pontryagin model …
Traceback (most recent call last):
  File "/content/pontryagin/experiments/embedding_energy.py", line 258, in <module>
    main()
  File "/content/pontryagin/experiments/embedding_energy.py", line 227, in main
    model = _load_model(results_dir=results_dir).to(device)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/pontryagin/experiments/embedding_energy.py", line 52, in _load_model
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
FileNotFoundError: Checkpoint not found: /content/drive/MyDrive/____UTP/Research/PHd/___RR_Pontryagin_embedded/results/fcn/pontryagin/best_model.pth


### 4d · Comparison across FCN models

In [11]:
!python experiments/compare_gradcam.py \
    --results-dir "{DRIVE_FCN_RESULTS}"


No metrics found. Run interpretability_analysis.py first.


## 5 · UNet backbone experiments
Models: `vanilla` | `euclidean` | `hyperbolic` | `pontryagin`

### 5a · HPO  *(optional)*

In [12]:
UNET_MODEL = 'pontryagin'   # change as needed
for UNET_MODEL in ['euclidean', 'hyperbolic', 'pontryagin']:
        !python experiments/tune_seg_uav_unet.py \
            --model        {UNET_MODEL} \
            --n-calls      30 \
            --trial-epochs 12 \
            --data-root    "{DRIVE_DATA_ROOT}" \
            --results-dir  "{DRIVE_UNET_RESULTS}"


── Trial 1 ──────────────────────────
  params: {'lr': 0.002453, 'weight_decay': 4e-06, 'unet_depth': np.int64(3)}
Traceback (most recent call last):
  File "/content/pontryagin/experiments/tune_seg_uav_unet.py", line 160, in <module>
    main()
  File "/content/pontryagin/experiments/tune_seg_uav_unet.py", line 125, in main
    result = forest_minimize(
             ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/skopt/optimizer/forest.py", line 203, in forest_minimize
    return base_minimize(
           ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/skopt/optimizer/base.py", line 332, in base_minimize
    next_y = func(next_x)
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/skopt/utils.py", line 779, in wrapper
    objective_value = func(**arg_dict)
                      ^^^^^^^^^^^^^^^^
  File "/content/pontryagin/experiments/tune_seg_uav_unet.py", line 105, in objective
    result = train_one_model(
             ^^^^^^^^^

### 5b · Full training

In [13]:
# Train all four UNet models sequentially.
# To train only one: replace the list with a single model name.
for UNET_MODEL in ['vanilla', 'euclidean', 'hyperbolic', 'pontryagin']:
    !python experiments/run_seg_uav_unet.py \
        --model       "{UNET_MODEL}" \
        --data-root   "{DRIVE_DATA_ROOT}" \
        --results-dir "{DRIVE_UNET_RESULTS}"



[VANILLA UNet] trainable params: 19,590,466
  ep   1  loss=0.5825  val_mIoU=0.4727  best=0.4727
  ep   5  loss=0.3750  val_mIoU=0.5356  best=0.5356
  ep  10  loss=0.3089  val_mIoU=0.6601  best=0.7291
  ep  15  loss=0.2101  val_mIoU=0.7606  best=0.7944
  ep  20  loss=0.2329  val_mIoU=0.7955  best=0.8157
  ep  25  loss=0.1723  val_mIoU=0.8491  best=0.8491
  ep  30  loss=0.1674  val_mIoU=0.8463  best=0.8649
  ep  35  loss=0.1574  val_mIoU=0.8570  best=0.8649
  ep  40  loss=0.1441  val_mIoU=0.8418  best=0.8649
  ep  45  loss=0.1308  val_mIoU=0.8599  best=0.8706
  ep  50  loss=0.1446  val_mIoU=0.8474  best=0.8706
  ep  55  loss=0.3072  val_mIoU=0.8623  best=0.8760
  ep  60  loss=0.1380  val_mIoU=0.8678  best=0.8760
  ep  65  loss=0.1454  val_mIoU=0.8276  best=0.8830
  ep  70  loss=0.1171  val_mIoU=0.8155  best=0.8830
  ep  75  loss=0.1088  val_mIoU=0.8364  best=0.8830
  Early stop at epoch 76
  Training time: 387s

  Test  mIoU=0.8780  pixel_acc=0.9658  macro_dice=0.9330

[EUCLIDEAN UNet] 

### 5c · Interpretability

In [14]:
!python experiments/interpretability_unet.py \
    --model       all \
    --data-root   "{DRIVE_DATA_ROOT}" \
    --results-dir "{DRIVE_UNET_RESULTS}"


── VANILLA UNet ──────────────────────────────
Traceback (most recent call last):
  File "/content/pontryagin/experiments/interpretability_unet.py", line 343, in <module>
    main()
  File "/content/pontryagin/experiments/interpretability_unet.py", line 333, in main
    analyse_model(mt, device,
  File "/content/pontryagin/experiments/interpretability_unet.py", line 249, in analyse_model
    cam_bg = ssc.generate_cam(inp, class_idx=0)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/pontryagin/experiments/interpretability_unet.py", line 116, in generate_cam
    cam = (feature_maps[0] * weights[:, None, None]).sum(dim=0)  # (H', W')
           ~~~~~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~~~~~
RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!


### 5d · Comparison across UNet models

In [15]:
!python experiments/compare_unet.py \
    --results-dir "{DRIVE_UNET_RESULTS}"


=== Segmentation metrics ===
  [skip] vanilla: no metrics.json
  [skip] euclidean: no metrics.json
  [skip] hyperbolic: no metrics.json
  [skip] pontryagin: no metrics.json

=== SegScoreCAM interpretability metrics ===
  [skip] vanilla: no gradcam/metrics.json
  [skip] euclidean: no gradcam/metrics.json
  [skip] hyperbolic: no gradcam/metrics.json
  [skip] pontryagin: no gradcam/metrics.json


## 6 · Download / inspect results

In [16]:
import json, pathlib, pandas as pd

print('=== FCN results ===')
for mt in ('euclidean', 'hyperbolic', 'pontryagin'):
    p = pathlib.Path(DRIVE_FCN_RESULTS) / mt / 'metrics.json'
    if p.exists():
        m = json.loads(p.read_text())
        print(f'  {mt:<12}  mIoU={m["macro_iou"]:.4f}  '
              f'Dice={m["macro_dice"]:.4f}  '
              f'pixel_acc={m["pixel_acc"]:.4f}')

print('\n=== UNet results ===')
for mt in ('vanilla', 'euclidean', 'hyperbolic', 'pontryagin'):
    p = pathlib.Path(DRIVE_UNET_RESULTS) / mt / 'metrics.json'
    if p.exists():
        m = json.loads(p.read_text())
        print(f'  {mt:<12}  mIoU={m["macro_iou"]:.4f}  '
              f'Dice={m["macro_dice"]:.4f}  '
              f'pixel_acc={m["pixel_acc"]:.4f}')

=== FCN results ===

=== UNet results ===
  vanilla       mIoU=0.8780  Dice=0.9330  pixel_acc=0.9658
  euclidean     mIoU=0.8479  Dice=0.9143  pixel_acc=0.9565
  hyperbolic    mIoU=0.8518  Dice=0.9169  pixel_acc=0.9575
  pontryagin    mIoU=0.8680  Dice=0.9270  pixel_acc=0.9614


In [17]:
# Zip and download all results to your local machine (optional)
import shutil
shutil.make_archive('/content/results_fcn',  'zip', DRIVE_FCN_RESULTS)
shutil.make_archive('/content/results_unet', 'zip', DRIVE_UNET_RESULTS)

from google.colab import files
files.download('/content/results_fcn.zip')
files.download('/content/results_unet.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>